In [3]:
from trachoma.trachoma_functions import *
import multiprocessing
num_cores = multiprocessing.cpu_count()
import numpy as np

#############################################################################################################################
#############################################################################################################################

# initialize parameters, sim_params, and demography

params = {'N': 2500,
          'av_I_duration' : 2,
          'av_ID_duration':200/7,
          'inf_red':0.45,
          'min_ID':11, #Parameters relating to duration of infection period, including ID period
          'av_D_duration':300/7,
          'min_D':1, #Parameters relating to duration of disease period
          'v_1':1,
          'v_2':2.6,
          'phi':1.4,
          'epsilon':0.5,#Parameters relating to lambda function- calculating force of infection
          #Parameters relating to MDA
          'MDA_Cov':0.8,
          'MDA_Eff': 0.85, # Efficacy of treatment
          'rho':0.3,
          'nweeks_year':52,
          'babiesMaxAge':0.5, #Note this is years, need to check it converts to weeks later
          'youngChildMaxAge':9,#Note this is years, need to check it converts to weeks later
          'olderChildMaxAge':15, #Note this is years, need to check it converts to weeks later
          'b1':1,#this relates to bacterial load function
          'ep2':0.114,
          'n_inf_sev':38,
          'TestSensitivity': 0.96,
          'TestSpecificity': 0.965,
          'SecularTrendIndicator': 0,
          'SecularTrendYearlyBetaDecrease': 0.01,
          'vacc_prob_block_transmission':  0.2, 
          'vacc_reduce_bacterial_load': 0.1, 
          'vacc_reduce_duration': 0.1,  
          'vacc_waning_length': 52 * 5}

burnin = 100*52
timesim = burnin + 13*52

sim_params = {'timesim': timesim, 
              'burnin': burnin,
              'N_MDA':5,
              'n_sim':80}


demog = {'tau': 0.0004807692, 
         'max_age': 3120,
         'mean_age': 1040}



previous_rounds = 0


Start_date = date(2018,1, 1)
End_date = date(2030,12,31)


In [23]:
def get_MDA_data(coverageFileName):
    MDAData = readPlatformData(coverageFileName, "MDA")
    MDA_dates = getInterventionDates(MDAData)
    MDA_times = get_Intervention_times(MDA_dates, Start_date, sim_params['burnin'])
    return MDAData, MDA_times

MDAData, MDA_times = get_MDA_data("scen1_no_interruption_vacc_every3_years.csv")

In [24]:
np.random.seed(10)
vals = Set_inits(params=params, demog=demog, sim_params = sim_params, MDAData = MDAData)    # Set initial conditions
vals = Seed_infection(params=params, vals=vals) # Seed infection
vals['IndI'] = np.ones(len(vals['IndI']))

In [25]:
MDA_times

array([5252, 5304, 5356, 5356, 5408, 5460, 5460, 5513, 5565, 5617, 5669,
       5721, 5774, 5826, 5878, 5930, 5982, 6034, 6087, 6139, 6191, 6243,
       6295, 6347])

In [28]:
MDAData

[[2019.0, 0, 100, 0.8, 0, 2],
 [2020.0, 0, 100, 0.3, 0, 2],
 [2021.0, 0, 100, 0.8, 0, 2],
 [2021.0, 1, 10, 0.4, 1, 2],
 [2022.0, 0, 100, 0.8, 0, 2],
 [2023.0, 0, 100, 0.8, 0, 2],
 [2023.0, 1, 10, 0.1, 1, 2],
 [2024.0, 0, 100, 0.8, 0, 2],
 [2025.0, 0, 100, 0.8, 0, 2],
 [2026.0, 0, 100, 0.8, 0, 2],
 [2027.0, 0, 100, 0.8, 0, 2],
 [2028.0, 0, 100, 0.8, 0, 2],
 [2029.0, 0, 100, 0.8, 0, 2],
 [2030.0, 0, 100, 0.8, 0, 2],
 [2031.0, 0, 100, 0.8, 0, 2],
 [2032.0, 0, 100, 0.8, 0, 2],
 [2033.0, 0, 100, 0.8, 0, 2],
 [2034.0, 0, 100, 0.8, 0, 2],
 [2035.0, 0, 100, 0.8, 0, 2],
 [2036.0, 0, 100, 0.8, 0, 2],
 [2037.0, 0, 100, 0.8, 0, 2],
 [2038.0, 0, 100, 0.8, 0, 2],
 [2039.0, 0, 100, 0.8, 0, 2],
 [2040.0, 0, 100, 0.8, 0, 2]]

In [67]:
nDoses = np.zeros(MDAData[0][-1], dtype=object)
coverage = np.zeros(MDAData[0][-1], dtype=object)
prevNMDA = np.zeros(MDAData[0][-1], dtype=object)
numMDA = np.zeros(MDAData[0][-1], dtype=object)

In [68]:
i=5356
nsims = 10000
for k in range(nsims):
    MDA_round = np.where(MDA_times == i)[0]
    for l in range(len(MDA_round)):
        MDA_round_current = MDA_round[l]
        ageStart, ageEnd, cov, systematic_non_compliance = get_MDA_params(MDAData, MDA_round_current, vals)
        vals = check_if_we_need_to_redraw_probability_of_treatment(cov, systematic_non_compliance, vals)
        out = MDA_timestep_Age_range(vals, params, ageStart, ageEnd)
        vals = out[0]
        nDoses, numMDA, coverage = update_MDA_information_for_output(MDAData, MDA_round_current, out,
                                                                        vals, ageStart, ageEnd, nDoses, numMDA, coverage)

In [69]:
coverage

array([7999.656799999917, 3864.5281690140646], dtype=object)